# Reproducible analysis: PONV after total knee arthroplasty

This notebook accompanies the manuscript **“Prediction of postoperative nausea and vomiting within 24 hours after total knee arthroplasty using interpretable machine learning.”**

The patient-level study data are not included. Place an institutionally authorized analytic file at `data/ponv_tka_input.xlsx`, using the schema in `data_dictionary.csv`. All reported models use fixed hyperparameter settings and a random seed of 42; no data-driven hyperparameter search was performed.


## Environment

Create a Python 3.11 environment and install the packages in `requirements.txt`. In Colab, uncomment and run the next line.


In [ ]:
# %pip install -r requirements.txt


## Load and validate the analytic data


In [ ]:
from pathlib import Path
import platform
import sys
import pandas as pd
import numpy as np

DATA_PATH = Path("data/ponv_tka_input.xlsx")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Input data not found: {DATA_PATH}. See data_dictionary.csv and example_input.csv."
    )

df = pd.read_excel(DATA_PATH)
print(f"Analytic rows: {len(df):,}; columns: {df.shape[1]}")
print(f"Python: {sys.version.split()[0]}; platform: {platform.platform()}")


In [ ]:
required_cols = [
    "PONV_24h",
    "age", "sex", "bmi", "smoking_status",
    "hypertension", "diabetes",
    "cardiovascular_disease", "cerebrovascular_disease",
    "chronic_lung_disease", "chronic_kidney_disease",
    "hx_PONV_or_motion", "ADR_analgesic_NV",
    "anesthesia_type", "LIA_use", "dexamethasone_use"
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing columns:", missing)


In [ ]:
# Standardize binary and categorical coding before model fitting.
# Sex is coded as female=1 so the model feature matches the manuscript label.
df["sex"] = df["sex"].replace({
    "F": 1, "Female": 1, "female": 1,
    "M": 0, "Male": 0, "male": 0,
})
df["sex"] = pd.to_numeric(df["sex"], errors="coerce").astype("Int64")

df["smoking_status"] = df["smoking_status"].replace({0: "No", 1: "Yes"})
df["hx_PONV_or_motion"] = df["hx_PONV_or_motion"].replace({0: "No", 1: "Yes"})

# General anesthesia is coded as 1; spinal/regional anesthesia is coded as 0.
df["anesthesia_type"] = df["anesthesia_type"].replace({
    "general": 1, "General": 1, "GA": 1,
    "spinal": 0, "Spinal": 0, "regional": 0, "Regional": 0,
})
df["anesthesia_type"] = pd.to_numeric(
    df["anesthesia_type"], errors="coerce"
).astype("Int64")

binary_cols = [
    "PONV_24h", "hypertension", "diabetes",
    "cardiovascular_disease", "cerebrovascular_disease",
    "chronic_lung_disease", "chronic_kidney_disease",
    "ADR_analgesic_NV", "LIA_use", "dexamethasone_use",
]
for column in binary_cols:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")

if df[required_cols].isna().any().any():
    raise ValueError(
        "Missing or unrecognized values remain after coding. "
        "Review data_dictionary.csv and the source data."
    )


In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def plot_calibration(y_true, y_prob, title):
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    brier = brier_score_loss(y_true, y_prob)

    plt.figure(figsize=(6,6))
    plt.plot(mean_pred, frac_pos, marker='o', label=f"Model (Brier={brier:.3f})")
    plt.plot([0,1], [0,1], linestyle='--', label="Ideal")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed probability")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return brier


In [ ]:
df["apfel_score"] = (
    (df["sex"] == 1).astype(int) +
    (df["hx_PONV_or_motion"] == "Yes").astype(int) +
    (df["smoking_status"] == "No").astype(int) +
    1
)


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) Output directory
# =========================
OUT_DIR = str(Path("outputs"))
os.makedirs(OUT_DIR, exist_ok=True)

TARGET = "PONV_24h"
SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5

# =========================
# 1) Feature sets (Model 1/2/3)
# =========================
MODEL_FEATURES = {
    "Model1": [
        "age","sex","bmi","smoking_status",
        "hypertension","diabetes",
        "cardiovascular_disease","cerebrovascular_disease",
        "chronic_lung_disease","chronic_kidney_disease"
    ],
    "Model2": [
        "age","sex","bmi","smoking_status",
        "hypertension","diabetes",
        "cardiovascular_disease","cerebrovascular_disease",
        "chronic_lung_disease","chronic_kidney_disease",
        "hx_PONV_or_motion","ADR_analgesic_NV"
    ],
    "Model3": [
        "age","sex","bmi","smoking_status",
        "hypertension","diabetes",
        "cardiovascular_disease","cerebrovascular_disease",
        "chronic_lung_disease","chronic_kidney_disease",
        "hx_PONV_or_motion","ADR_analgesic_NV",
        "anesthesia_type","LIA_use","dexamethasone_use"
    ],
}

def _check_columns(df: pd.DataFrame):
    missing = []
    for feats in MODEL_FEATURES.values():
        for c in feats:
            if c not in df.columns:
                missing.append(c)
    if TARGET not in df.columns:
        missing.append(TARGET)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(set(missing))}")

# =========================
# 2) Models
# =========================
def get_models(seed=42):
    return {
        "Logistic": LogisticRegression(max_iter=3000),
        "XGBoost": XGBClassifier(
            n_estimators=500,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=900,
            max_depth=6,
            min_samples_leaf=20,
            random_state=seed,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=800,
            depth=4,
            learning_rate=0.05,
            loss_function="Logloss",
            verbose=False,
            random_seed=seed,
        ),
    }

# =========================
# 3) Preprocess
# =========================
def build_preprocess(X: pd.DataFrame):
    num_vars = X.select_dtypes(include=["int64","float64","bool"]).columns.tolist()
    cat_vars = [c for c in X.columns if c not in num_vars]

    return ColumnTransformer([
        ("num", "passthrough", num_vars),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_vars),
    ])

# =========================
# 4) Main runner (NO Sens/Spec)
# =========================
def run_traincv_test_metrics(df: pd.DataFrame, seed=42, test_size=0.2, n_splits=5):
    _check_columns(df)

    y_all = df[TARGET].astype(int).values
    idx = np.arange(len(df))

    idx_train, idx_test = train_test_split(
        idx, test_size=test_size, stratify=y_all, random_state=seed
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed)

    rows_perf = []
    rows_test_pred = []

    for featset_name, feats in MODEL_FEATURES.items():
        X = df[feats]
        X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
        y_train, y_test = y_all[idx_train], y_all[idx_test]

        preprocess = build_preprocess(X_train)

        for model_name, estimator in models.items():
            pipe = Pipeline([("prep", preprocess), ("clf", estimator)])

            cv_res = cross_validate(
                pipe, X_train, y_train,
                cv=cv,
                scoring={"auc": "roc_auc", "prauc": "average_precision"},
                n_jobs=-1
            )

            cv_auc_m, cv_auc_s = cv_res["test_auc"].mean(), cv_res["test_auc"].std(ddof=1)
            cv_pr_m,  cv_pr_s  = cv_res["test_prauc"].mean(), cv_res["test_prauc"].std(ddof=1)

            pipe.fit(X_train, y_train)
            test_prob = pipe.predict_proba(X_test)[:, 1]

            test_auc = roc_auc_score(y_test, test_prob)
            test_pr  = average_precision_score(y_test, test_prob)

            rows_perf.append({
                "FeatureSet": featset_name,
                "Model": model_name,
                "TrainCV_AUC (mean±SD)": f"{cv_auc_m:.3f} ± {cv_auc_s:.3f}",
                "TrainCV_PR-AUC (mean±SD)": f"{cv_pr_m:.3f} ± {cv_pr_s:.3f}",
                "Test_AUC": f"{test_auc:.3f}",
                "Test_PR-AUC": f"{test_pr:.3f}",
            })

            rows_test_pred.append(pd.DataFrame({
                "FeatureSet": featset_name,
                "Model": model_name,
                "index": idx_test,
                "y_true": y_test,
                "y_prob": test_prob
            }))

    df_perf = pd.DataFrame(rows_perf)
    df_test_pred = pd.concat(rows_test_pred, ignore_index=True)

    order_models = ["Logistic", "XGBoost", "RandomForest", "CatBoost"]

    def wide(col):
        return df_perf.pivot(index="Model", columns="FeatureSet", values=col)\
                      .reindex(order_models).reset_index()

    return (
        df_perf,
        df_test_pred,
        wide("TrainCV_AUC (mean±SD)"),
        wide("TrainCV_PR-AUC (mean±SD)"),
        wide("Test_AUC"),
        wide("Test_PR-AUC"),
    )

# =========================
# 5) Run + Save
# =========================
(df_perf, df_test_pred,
 table_traincv_auc, table_traincv_prauc,
 table_test_auc, table_test_prauc) = run_traincv_test_metrics(
    df, seed=SEED, test_size=TEST_SIZE, n_splits=N_SPLITS
)

df_perf.to_csv(os.path.join(OUT_DIR, "perf_trainCV_AUC_PRAUC_and_test.csv"), index=False)
df_test_pred.to_csv(os.path.join(OUT_DIR, "test_predictions_probabilities.csv"), index=False)

table_traincv_auc.to_csv(os.path.join(OUT_DIR, "table_trainCV_AUC_wide.csv"), index=False)
table_traincv_prauc.to_csv(os.path.join(OUT_DIR, "table_trainCV_PRAUC_wide.csv"), index=False)
table_test_auc.to_csv(os.path.join(OUT_DIR, "table_test_AUC_wide.csv"), index=False)
table_test_prauc.to_csv(os.path.join(OUT_DIR, "table_test_PRAUC_wide.csv"), index=False)

display(table_traincv_auc)
display(table_traincv_prauc)
display(table_test_auc)
display(table_test_prauc)


In [ ]:
# test index 가져오기
test_idx = df_test_pred["index"].unique()

# 원본 df에서 test set 추출
test_df = df.iloc[test_idx].copy()

# Apfel score 계산 (이미 df에 있으므로 그대로 사용 가능)
from sklearn.metrics import roc_auc_score

apfel_prob = test_df["apfel_score"] / 4.0
apfel_auc = roc_auc_score(test_df["PONV_24h"], apfel_prob)

print("Apfel AUC:", round(apfel_auc, 3))


In [ ]:
# RandomForest Model3만 예시
rf_model3 = df_test_pred[
    (df_test_pred["Model"] == "RandomForest") &
    (df_test_pred["FeatureSet"] == "Model3")
]

ml_auc = roc_auc_score(rf_model3["y_true"], rf_model3["y_prob"])

print("ML AUC:", round(ml_auc, 3))
print("Apfel AUC:", round(apfel_auc, 3))


In [ ]:
def plot_calibration(y_true, y_prob, title):
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    brier = brier_score_loss(y_true, y_prob)

    plt.figure(figsize=(6,6))
    plt.plot(mean_pred, frac_pos, marker='o',
             label=f"Model (Brier score = {brier:.3f})")
    plt.plot([0,1], [0,1], linestyle='--', label="Ideal")

    plt.xlabel("Predicted probability")
    plt.ylabel("Observed probability")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return brier


In [ ]:
brier_ml = plot_calibration(
    rf_model3["y_true"],
    rf_model3["y_prob"],
    "Random Forest (Model 3)"
)

brier_apfel = plot_calibration(
    test_df["PONV_24h"],
    apfel_prob,
    "Apfel Score"
)

print("ML Brier:", brier_ml)
print("Apfel Brier:", brier_apfel)


In [ ]:
# ML calibration
plot_calibration(
    rf_model3["y_true"],
    rf_model3["y_prob"],
    "Random Forest (Model 3)"
)

# Apfel calibration
plot_calibration(
    test_df["PONV_24h"],
    apfel_prob,
    "Apfel Score"
)


In [ ]:
def decision_curve(y_true, y_prob, thresholds):
    n = len(y_true)
    net_benefit = []

    for t in thresholds:
        pred = (y_prob >= t).astype(int)

        TP = ((pred == 1) & (y_true == 1)).sum()
        FP = ((pred == 1) & (y_true == 0)).sum()

        nb = (TP / n) - (FP / n) * (t / (1 - t))
        net_benefit.append(nb)

    return np.array(net_benefit)


In [ ]:
thresholds = np.linspace(0.01, 0.5, 100)

nb_ml = decision_curve(rf_model3["y_true"].values, rf_model3["y_prob"].values, thresholds)
nb_apfel = decision_curve(test_df["PONV_24h"].values, apfel_prob.values, thresholds)

plt.figure(figsize=(7,5))
plt.plot(thresholds, nb_ml, label="ML model")
plt.plot(thresholds, nb_apfel, label="Apfel")
plt.plot(thresholds, np.zeros_like(thresholds), linestyle="--", label="Treat none")

prevalence = test_df["PONV_24h"].mean()
nb_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))
plt.plot(thresholds, nb_all, linestyle=":", label="Treat all")

plt.xlabel("Threshold probability")
plt.ylabel("Net benefit")
plt.legend()
plt.title("Decision Curve Analysis")
plt.show()


## Temporal trend and temporal validation sensitivity analysis

This section addresses temporal bias across the 2015-2025 study period. It summarizes annual PONV incidence, creates a chronological development/validation split, evaluates the final random forest Step 3 model in later patients, and exports manuscript-ready tables and figures. The current data use `surgery_year`; if an exact operation date is later available, it can instead be specified with `SURGERY_DATE_COL` in the first added code cell.

In [ ]:
"""Temporal trend and temporal validation sensitivity analysis for PONV_TKA2.

Append this code after the existing model-development cells. It expects the
existing notebook to have already created df, MODEL_FEATURES, build_preprocess,
get_models, TARGET, SEED, and OUT_DIR.
"""

import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import norm
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline


# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------
# If automatic detection fails, enter the exact date-column name here, e.g.
# SURGERY_DATE_COL = "surgery_date"
SURGERY_DATE_COL = None
SURGERY_YEAR_COL = "surgery_year"
TEMPORAL_TEST_SIZE = 0.20
N_BOOTSTRAP = 2000
N_CALIBRATION_BINS = 8

DATE_CANDIDATES = [
    "surgery_date",
    "operation_date",
    "op_date",
    "date_of_surgery",
    "surgery_datetime",
    "operation_datetime",
    "수술일",
    "수술일자",
]


def detect_date_column(data, specified=None):
    """Return a valid surgery-date column or raise an informative error."""
    if specified is not None:
        if specified not in data.columns:
            raise ValueError(
                f"SURGERY_DATE_COL='{specified}' is not present in the data. "
                f"Available columns: {list(data.columns)}"
            )
        return specified

    lower_map = {str(c).strip().lower(): c for c in data.columns}
    for candidate in DATE_CANDIDATES:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    fuzzy = [
        c for c in data.columns
        if ("date" in str(c).lower())
        and any(k in str(c).lower() for k in ["surg", "oper", "op_"])
    ]
    if len(fuzzy) == 1:
        return fuzzy[0]

    raise ValueError(
        "A surgery-date column could not be identified automatically. "
        "Set SURGERY_DATE_COL to the exact column name and rerun this cell. "
        f"Possible date-like columns: {fuzzy}"
    )


def wilson_ci(events, total, alpha=0.05):
    """Wilson confidence interval for a binomial proportion."""
    if total == 0:
        return np.nan, np.nan
    z = norm.ppf(1 - alpha / 2)
    p = events / total
    denom = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denom
    half = z * np.sqrt((p * (1 - p) / total) + z**2 / (4 * total**2)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def stratified_bootstrap_ci(y_true, y_prob, metric_func, n_boot=2000, seed=42):
    """Percentile bootstrap CI while retaining both outcome classes."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    rng = np.random.default_rng(seed)
    idx0 = np.flatnonzero(y_true == 0)
    idx1 = np.flatnonzero(y_true == 1)
    estimates = []

    if len(idx0) == 0 or len(idx1) == 0:
        return np.nan, np.nan

    for _ in range(n_boot):
        sampled = np.concatenate([
            rng.choice(idx0, size=len(idx0), replace=True),
            rng.choice(idx1, size=len(idx1), replace=True),
        ])
        rng.shuffle(sampled)
        try:
            estimates.append(metric_func(y_true[sampled], y_prob[sampled]))
        except ValueError:
            continue

    if not estimates:
        return np.nan, np.nan
    return tuple(np.quantile(estimates, [0.025, 0.975]))


def logistic_mle(y_true, design_matrix):
    """Fit a small logistic model and return coefficients and standard errors."""
    y_true = np.asarray(y_true, dtype=float)
    design_matrix = np.asarray(design_matrix, dtype=float)

    def negative_log_likelihood(beta):
        linear_predictor = design_matrix @ beta
        return np.sum(
            np.logaddexp(0.0, linear_predictor)
            - y_true * linear_predictor
        )

    fit = minimize(
        negative_log_likelihood,
        x0=np.zeros(design_matrix.shape[1]),
        method="BFGS",
    )
    if not fit.success:
        warnings.warn(f"Logistic calibration fit warning: {fit.message}")

    fitted_probability = expit(design_matrix @ fit.x)
    weights = fitted_probability * (1 - fitted_probability)
    hessian = design_matrix.T @ (design_matrix * weights[:, None])
    covariance = np.linalg.pinv(hessian)
    standard_error = np.sqrt(np.clip(np.diag(covariance), 0, None))
    return fit.x, standard_error


def calibration_intercept_slope(y_true, y_prob):
    """Estimate calibration intercept and slope from logit predictions."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.clip(np.asarray(y_prob, dtype=float), 1e-6, 1 - 1e-6)
    logit_prob = np.log(y_prob / (1 - y_prob))
    design = np.column_stack([np.ones(len(y_true)), logit_prob])
    coefficients, standard_error = logistic_mle(y_true, design)
    z = norm.ppf(0.975)
    return {
        "Calibration_intercept": float(coefficients[0]),
        "Calibration_intercept_CI_low": float(coefficients[0] - z * standard_error[0]),
        "Calibration_intercept_CI_high": float(coefficients[0] + z * standard_error[0]),
        "Calibration_slope": float(coefficients[1]),
        "Calibration_slope_CI_low": float(coefficients[1] - z * standard_error[1]),
        "Calibration_slope_CI_high": float(coefficients[1] + z * standard_error[1]),
    }


In [ ]:
# ---------------------------------------------------------------------------
# 1) Prepare year/date variable and describe annual temporal trends
# ---------------------------------------------------------------------------
temporal_df = df.copy()

if SURGERY_YEAR_COL in temporal_df.columns:
    year_numeric = pd.to_numeric(
        temporal_df[SURGERY_YEAR_COL], errors="coerce"
    )
    invalid_year = (
        year_numeric.isna()
        | (year_numeric % 1 != 0)
        | ~year_numeric.between(2015, 2025)
    )
    if invalid_year.any():
        raise ValueError(
            f"{int(invalid_year.sum())} records have invalid surgery years. "
            "Expected integer values from 2015 through 2025."
        )
    temporal_df["surgery_year"] = year_numeric.astype(int)
    # A mid-year placeholder is used only for chronological sorting. Because
    # every patient in a calendar year receives the same placeholder, no year
    # can be split between development and validation cohorts.
    temporal_df["_surgery_date"] = pd.to_datetime(
        temporal_df["surgery_year"].astype(str) + "-07-01"
    )
    TEMPORAL_RESOLUTION = "year"
    DATE_COL = SURGERY_YEAR_COL
    print("Surgery-year column:", DATE_COL)
else:
    DATE_COL = detect_date_column(temporal_df, SURGERY_DATE_COL)
    print("Surgery-date column:", DATE_COL)
    temporal_df["_surgery_date"] = pd.to_datetime(
        temporal_df[DATE_COL], errors="coerce"
    )
    missing_dates = int(temporal_df["_surgery_date"].isna().sum())
    if missing_dates:
        raise ValueError(
            f"{missing_dates} records have missing or invalid surgery dates. "
            "Resolve these dates before temporal validation; records were not "
            "silently excluded."
        )
    temporal_df["surgery_year"] = temporal_df["_surgery_date"].dt.year.astype(int)
    TEMPORAL_RESOLUTION = "date"

temporal_df[TARGET] = pd.to_numeric(
    temporal_df[TARGET], errors="raise"
).astype(int)

annual = (
    temporal_df.groupby("surgery_year", as_index=False)
    .agg(
        Total_patients=(TARGET, "size"),
        PONV_events=(TARGET, "sum"),
        PONV_incidence=(TARGET, "mean"),
    )
)

annual_ci = annual.apply(
    lambda row: wilson_ci(int(row["PONV_events"]), int(row["Total_patients"])),
    axis=1,
)
annual["PONV_CI_low"] = [x[0] for x in annual_ci]
annual["PONV_CI_high"] = [x[1] for x in annual_ci]

# Add annual management-practice rates when available.
if "anesthesia_type" in temporal_df.columns:
    temporal_df["_general_anesthesia"] = (
        pd.to_numeric(temporal_df["anesthesia_type"], errors="coerce") == 1
    ).astype(int)

management_candidates = {
    "General_anesthesia_rate": "_general_anesthesia",
    "LIA_use_rate": "LIA_use",
    "Dexamethasone_use_rate": "dexamethasone_use",
}

pca_candidates = [
    "PCA_use", "pca_use", "postoperative_PCA_use", "postop_PCA_use",
    "opioid_PCA_use"
]
pca_col = next((c for c in pca_candidates if c in temporal_df.columns), None)
if pca_col is not None:
    management_candidates["PCA_use_rate"] = pca_col

for output_name, source_col in management_candidates.items():
    if source_col in temporal_df.columns:
        numeric = pd.to_numeric(temporal_df[source_col], errors="coerce")
        if numeric.notna().any():
            temporal_df[f"_numeric_{source_col}"] = numeric
            yearly_rate = temporal_df.groupby("surgery_year")[
                f"_numeric_{source_col}"
            ].mean()
            annual[output_name] = annual["surgery_year"].map(yearly_rate)

annual_display = annual.copy()
for col in [c for c in annual_display.columns if "incidence" in c.lower() or "rate" in c.lower() or "CI_" in c]:
    annual_display[col] = annual_display[col].map(
        lambda x: f"{100*x:.1f}%" if pd.notna(x) else ""
    )

display(annual_display)

# Descriptive test for a linear calendar-year trend in PONV incidence.
year_centered = temporal_df["surgery_year"] - temporal_df["surgery_year"].mean()
trend_design = np.column_stack([
    np.ones(len(temporal_df)),
    year_centered.to_numpy(dtype=float),
])
trend_coefficients, trend_standard_error = logistic_mle(
    temporal_df[TARGET].to_numpy(dtype=int), trend_design
)
trend_z = norm.ppf(0.975)
trend_p_value = 2 * (
    1 - norm.cdf(abs(trend_coefficients[1] / trend_standard_error[1]))
)
annual_trend = pd.DataFrame([{
    "Odds_ratio_per_calendar_year": np.exp(trend_coefficients[1]),
    "CI_low": np.exp(trend_coefficients[1] - trend_z * trend_standard_error[1]),
    "CI_high": np.exp(trend_coefficients[1] + trend_z * trend_standard_error[1]),
    "P_value": trend_p_value,
}])
display(annual_trend)


In [ ]:
# ---------------------------------------------------------------------------
# 2) Chronological development/validation split
# ---------------------------------------------------------------------------
temporal_df = temporal_df.sort_values("_surgery_date", kind="mergesort").copy()
n_total = len(temporal_df)
n_development_target = int(np.floor(n_total * (1 - TEMPORAL_TEST_SIZE)))
candidate_cutoff = temporal_df["_surgery_date"].iloc[n_development_target - 1]

# Keep all operations on the same calendar date in one cohort.
development_df = temporal_df[
    temporal_df["_surgery_date"] <= candidate_cutoff
].copy()
validation_df = temporal_df[
    temporal_df["_surgery_date"] > candidate_cutoff
].copy()

if validation_df.empty:
    raise ValueError("The chronological split produced an empty validation cohort.")
if development_df[TARGET].nunique() < 2 or validation_df[TARGET].nunique() < 2:
    raise ValueError(
        "Both temporal cohorts must contain patients with and without PONV. "
        "Consider a different prespecified cutoff after inspecting annual counts."
    )

actual_temporal_fraction = len(validation_df) / n_total
if abs(actual_temporal_fraction - TEMPORAL_TEST_SIZE) > 0.02:
    warnings.warn(
        "Keeping all operations from the cutoff date in the same cohort made "
        f"the temporal validation fraction {actual_temporal_fraction:.1%}."
    )

if TEMPORAL_RESOLUTION == "year":
    dev_start = int(development_df["surgery_year"].min())
    dev_end = int(development_df["surgery_year"].max())
    val_start = int(validation_df["surgery_year"].min())
    val_end = int(validation_df["surgery_year"].max())
else:
    dev_start = development_df["_surgery_date"].min().date()
    dev_end = development_df["_surgery_date"].max().date()
    val_start = validation_df["_surgery_date"].min().date()
    val_end = validation_df["_surgery_date"].max().date()

cohort_summary = pd.DataFrame([
    {
        "Cohort": "Temporal development",
        "Start_period": dev_start,
        "End_period": dev_end,
        "N": len(development_df),
        "PONV_events": int(development_df[TARGET].sum()),
        "PONV_incidence": development_df[TARGET].mean(),
    },
    {
        "Cohort": "Temporal validation",
        "Start_period": val_start,
        "End_period": val_end,
        "N": len(validation_df),
        "PONV_events": int(validation_df[TARGET].sum()),
        "PONV_incidence": validation_df[TARGET].mean(),
    },
])
display(cohort_summary)


In [ ]:
# ---------------------------------------------------------------------------
# 3) Train RF Model 3 in earlier patients and evaluate unchanged in later ones
# ---------------------------------------------------------------------------
temporal_features = MODEL_FEATURES["Model3"]
X_dev = development_df[temporal_features]
y_dev = development_df[TARGET].astype(int).to_numpy()
X_val = validation_df[temporal_features]
y_val = validation_df[TARGET].astype(int).to_numpy()

temporal_rf = Pipeline([
    ("prep", build_preprocess(X_dev)),
    ("clf", get_models(SEED)["RandomForest"]),
])
temporal_rf.fit(X_dev, y_dev)
val_prob = temporal_rf.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, val_prob)
prauc = average_precision_score(y_val, val_prob)
brier = brier_score_loss(y_val, val_prob)

auc_ci = stratified_bootstrap_ci(
    y_val, val_prob, roc_auc_score, N_BOOTSTRAP, SEED
)
prauc_ci = stratified_bootstrap_ci(
    y_val, val_prob, average_precision_score, N_BOOTSTRAP, SEED + 1
)
brier_ci = stratified_bootstrap_ci(
    y_val, val_prob, brier_score_loss, N_BOOTSTRAP, SEED + 2
)

temporal_metrics = {
    "Model": "RandomForest_Model3",
    "Development_N": len(development_df),
    "Development_events": int(y_dev.sum()),
    "Validation_N": len(validation_df),
    "Validation_events": int(y_val.sum()),
    "Validation_prevalence": float(y_val.mean()),
    "AUC": auc,
    "AUC_CI_low": auc_ci[0],
    "AUC_CI_high": auc_ci[1],
    "PR_AUC": prauc,
    "PR_AUC_CI_low": prauc_ci[0],
    "PR_AUC_CI_high": prauc_ci[1],
    "Brier_score": brier,
    "Brier_CI_low": brier_ci[0],
    "Brier_CI_high": brier_ci[1],
}
temporal_metrics.update(calibration_intercept_slope(y_val, val_prob))
temporal_metrics_df = pd.DataFrame([temporal_metrics])
display(temporal_metrics_df.T)

prediction_period = (
    validation_df["surgery_year"].to_numpy()
    if TEMPORAL_RESOLUTION == "year"
    else validation_df["_surgery_date"].dt.date.to_numpy()
)
temporal_predictions = pd.DataFrame({
    "original_index": validation_df.index,
    "surgery_period": prediction_period,
    "y_true": y_val,
    "y_prob": val_prob,
})


In [ ]:
# ---------------------------------------------------------------------------
# 4) Plots
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
lower_err = annual["PONV_incidence"] - annual["PONV_CI_low"]
upper_err = annual["PONV_CI_high"] - annual["PONV_incidence"]
ax.errorbar(
    annual["surgery_year"],
    annual["PONV_incidence"],
    yerr=[lower_err, upper_err],
    fmt="o-",
    capsize=4,
    color="#1f77b4",
)
ax.set_xlabel("Surgery year")
ax.set_ylabel("PONV incidence")
ax.set_title("Annual distribution of PONV incidence")
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{100*y:.0f}%"))
ax.grid(alpha=0.25)
fig.tight_layout()
annual_plot_png = os.path.join(OUT_DIR, "annual_PONV_incidence.png")
annual_plot_pdf = os.path.join(OUT_DIR, "annual_PONV_incidence.pdf")
fig.savefig(annual_plot_png, dpi=300, bbox_inches="tight")
fig.savefig(annual_plot_pdf, bbox_inches="tight")
plt.show()

fpr, tpr, _ = roc_curve(y_val, val_prob)
precision, recall, _ = precision_recall_curve(y_val, val_prob)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, lw=2, label=f"RF Model 3 (AUC={auc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("False-positive rate")
axes[0].set_ylabel("True-positive rate")
axes[0].set_title("Temporal validation ROC curve")
axes[0].legend(loc="lower right")

axes[1].plot(recall, precision, lw=2, label=f"RF Model 3 (PR-AUC={prauc:.3f})")
axes[1].axhline(y_val.mean(), linestyle="--", color="gray", label="Prevalence")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Temporal validation precision-recall curve")
axes[1].legend(loc="best")
fig.tight_layout()
roc_pr_png = os.path.join(OUT_DIR, "temporal_validation_ROC_PR.png")
roc_pr_pdf = os.path.join(OUT_DIR, "temporal_validation_ROC_PR.pdf")
fig.savefig(roc_pr_png, dpi=300, bbox_inches="tight")
fig.savefig(roc_pr_pdf, bbox_inches="tight")
plt.show()

frac_pos, mean_pred = calibration_curve(
    y_val, val_prob, n_bins=N_CALIBRATION_BINS, strategy="quantile"
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mean_pred, frac_pos, marker="o", lw=2, label="RF Model 3")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed probability")
ax.set_title("Temporal validation calibration plot")
ax.legend(loc="best")
fig.tight_layout()
calibration_png = os.path.join(OUT_DIR, "temporal_validation_calibration.png")
calibration_pdf = os.path.join(OUT_DIR, "temporal_validation_calibration.pdf")
fig.savefig(calibration_png, dpi=300, bbox_inches="tight")
fig.savefig(calibration_pdf, bbox_inches="tight")
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 5) Export manuscript-ready results
# ---------------------------------------------------------------------------
annual.to_csv(os.path.join(OUT_DIR, "annual_PONV_summary.csv"), index=False)
annual_trend.to_csv(os.path.join(OUT_DIR, "annual_PONV_trend.csv"), index=False)
cohort_summary.to_csv(os.path.join(OUT_DIR, "temporal_cohort_summary.csv"), index=False)
temporal_metrics_df.to_csv(
    os.path.join(OUT_DIR, "temporal_validation_metrics.csv"), index=False
)
temporal_predictions.to_csv(
    os.path.join(OUT_DIR, "temporal_validation_predictions.csv"), index=False
)

excel_path = os.path.join(OUT_DIR, "temporal_validation_results.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    annual.to_excel(writer, sheet_name="Annual PONV", index=False)
    annual_trend.to_excel(writer, sheet_name="Annual trend", index=False)
    cohort_summary.to_excel(writer, sheet_name="Cohort summary", index=False)
    temporal_metrics_df.to_excel(writer, sheet_name="Temporal metrics", index=False)
    temporal_predictions.to_excel(writer, sheet_name="Predictions", index=False)

print("Saved temporal-validation outputs to:", OUT_DIR)
print("Excel summary:", excel_path)


## Feature importance and SHAP analyses

The following analysis refits each model on the same training split and calculates feature importance and SHAP values in the held-out test set. Expanded one-hot encoded features are aggregated to their original clinical variables for the supplementary tables.


In [ ]:
# ============================================================
# Supplementary Tables:
# Feature importance + mean(|SHAP|) + direction of association
# (per model × per step) + ordered by your requested variable order
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import shap
from scipy.stats import spearmanr
from pandas.api.types import CategoricalDtype


# =========================
# 0) Settings
# =========================
OUT_DIR = str(Path("outputs"))
os.makedirs(OUT_DIR, exist_ok=True)

TARGET = "PONV_24h"
SEED = 42
TEST_SIZE = 0.2

SAVE_PREFIX = "suppTable_featureImp_SHAP_dir"  # <-- 파일 이름 prefix
TOPK = None  # e.g., 10 or 15 for top-k rows; None = keep all

# =========================
# 1) Feature sets (Step 1/2/3)
# =========================
MODEL_FEATURES = {
    "Step1": [
        "age", "sex", "bmi", "smoking_status",
        "hypertension", "diabetes",
        "cardiovascular_disease", "cerebrovascular_disease",
        "chronic_lung_disease", "chronic_kidney_disease"
    ],
    "Step2": [
        "age", "sex", "bmi", "smoking_status",
        "hypertension", "diabetes",
        "cardiovascular_disease", "cerebrovascular_disease",
        "chronic_lung_disease", "chronic_kidney_disease",
        "hx_PONV_or_motion", "ADR_analgesic_NV"
    ],
    "Step3": [
        "age", "sex", "bmi", "smoking_status",
        "hypertension", "diabetes",
        "cardiovascular_disease", "cerebrovascular_disease",
        "chronic_lung_disease", "chronic_kidney_disease",
        "hx_PONV_or_motion", "ADR_analgesic_NV",
        "anesthesia_type", "LIA_use", "dexamethasone_use"
    ],
}

# =========================
# 2) Display label mapping (원 변수명 -> 논문 표기)
#    (정렬/표시를 위해 중요)
# =========================
DISPLAY_NAME = {
    "age": "Age",
    "bmi": "BMI",
    "sex": "Sex, female",
    "smoking_status": "Smoking status, yes",
    "hypertension": "HTN, yes",
    "diabetes": "DM, yes",
    "cardiovascular_disease": "CVD, yes",
    "cerebrovascular_disease": "CeVD, yes",
    "chronic_lung_disease": "CLD, yes",
    "chronic_kidney_disease": "CKD, yes",
    "hx_PONV_or_motion": "History of PONV, yes",
    "ADR_analgesic_NV": "Analgesic-related ADR with NV",
    "anesthesia_type": "General anesthesia",   # <-- 아래 NOTE 참고
    "LIA_use": "LIA use",
    "dexamethasone_use": "Dexamethasone use",
}

# NOTE:
# - anesthesia_type이 "General/Spinal" 같은 범주형이면
#   OneHot 후 'anesthesia_type_General' 같은 expanded feature가 생깁니다.
#   이 코드는 이를 'anesthesia_type'로 묶어서 "General anesthesia"로 표시합니다.
# - 만약 anesthesia_type이 이미 0/1 (General anesthesia yes/no) 형태라면
#   더 깔끔하게 동작합니다.

FEATURE_ORDER = [
    "Age",
    "BMI",
    "Sex, female",
    "Smoking status, yes",
    "HTN, yes",
    "DM, yes",
    "CVD, yes",
    "CeVD, yes",
    "CLD, yes",
    "CKD, yes",
    "History of PONV, yes",
    "Analgesic-related ADR with NV",
    "General anesthesia",
    "LIA use",
    "Dexamethasone use",
]
FEATURE_ORDER_TYPE = CategoricalDtype(categories=FEATURE_ORDER, ordered=True)


# =========================
# 3) Sanity checks
# =========================
def _check_columns(df: pd.DataFrame):
    missing = []
    for feats in MODEL_FEATURES.values():
        for c in feats:
            if c not in df.columns:
                missing.append(c)
    if TARGET not in df.columns:
        missing.append(TARGET)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(set(missing))}")


# =========================
# 4) Models
# =========================
def get_models(seed=42):
    return {
        "LR": LogisticRegression(max_iter=3000),
        "XGBoost": XGBClassifier(
            n_estimators=500,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1,
        ),
        "RF": RandomForestClassifier(
            n_estimators=900,
            max_depth=6,
            min_samples_leaf=20,
            random_state=seed,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=800,
            depth=4,
            learning_rate=0.05,
            loss_function="Logloss",
            verbose=False,
            random_seed=seed,
        ),
    }


# =========================
# 5) Preprocess
# =========================
def build_preprocess(X: pd.DataFrame):
    # bool도 numeric passthrough
    num_vars = X.select_dtypes(include=["int64", "float64", "bool", "int32", "float32"]).columns.tolist()
    cat_vars = [c for c in X.columns if c not in num_vars]

    return ColumnTransformer([
        ("num", "passthrough", num_vars),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_vars),
    ])


def get_feature_names_and_groups(prep: ColumnTransformer, X: pd.DataFrame):
    """
    Returns:
      expanded_names: list[str]  (after preprocessing)
      group_map: dict[str, str]  expanded_feature -> original_feature
      num_vars, cat_vars (original feature names)
    """
    num_vars = list(prep.transformers_[0][2])
    cat_vars = list(prep.transformers_[1][2])
    ohe = prep.named_transformers_["cat"]

    expanded_names = []
    group_map = {}

    # numeric passthrough
    for c in num_vars:
        expanded_names.append(c)
        group_map[c] = c

    # categorical one-hot
    if len(cat_vars) > 0:
        ohe_names = ohe.get_feature_names_out(cat_vars).tolist()  # e.g., sex_M
        expanded_names.extend(ohe_names)

        # safest: match prefix against cat_vars
        for name in ohe_names:
            orig = None
            for v in cat_vars:
                if name.startswith(v + "_"):
                    orig = v
                    break
            if orig is None:
                orig = name.split("_")[0]
            group_map[name] = orig

    return expanded_names, group_map, num_vars, cat_vars


def _to_dense(M):
    return M.toarray() if hasattr(M, "toarray") else np.asarray(M)


# =========================
# 6) SHAP helpers
# =========================
def _ensure_shap_2d(sv):
    """
    Ensure SHAP values are 2D: (n_samples, n_features)
    Handles possible shapes:
      - list of arrays (binary classification): [class0, class1]
      - (n, p) already ok
      - (n, p, 2): take class 1
      - (n, 2, p): take class 1 (rare)
    """
    if isinstance(sv, list):
        sv = sv[1] if len(sv) > 1 else sv[0]

    sv = np.asarray(sv)

    if sv.ndim == 2:
        return sv

    if sv.ndim == 3:
        if sv.shape[2] == 2:      # (n, p, 2)
            return sv[:, :, 1]
        if sv.shape[1] == 2:      # (n, 2, p)
            return sv[:, 1, :]
        if sv.shape[2] == 1:
            return sv[:, :, 0]
        if sv.shape[1] == 1:
            return sv[:, 0, :]

    raise ValueError(f"Unexpected SHAP shape: {sv.shape}")


def compute_shap_2d(model_name: str, fitted_clf, X_expanded, background=None):
    """
    Returns:
      shap_values_2d: np.ndarray (n_samples, n_features) for class 1
    """
    Xd = _to_dense(X_expanded)

    if model_name in ["RF", "XGBoost", "CatBoost"]:
        explainer = shap.TreeExplainer(fitted_clf)
        sv = explainer.shap_values(Xd)
        return _ensure_shap_2d(sv)

    if model_name == "LR":
        if background is None:
            n_bg = min(200, Xd.shape[0])
            bg = Xd[:n_bg]
        else:
            bg = _to_dense(background)

        # SHAP 최신 버전에서 masker 권장 경고가 뜰 수 있으나 실행은 됩니다.
        explainer = shap.LinearExplainer(
            fitted_clf,
            bg,
            feature_perturbation="interventional"
        )
        sv = explainer.shap_values(Xd)
        return _ensure_shap_2d(sv)

    raise ValueError(f"Unknown model: {model_name}")


# =========================
# 7) Importance extractors (expanded-feature level)
# =========================
def get_importance_expanded(model_name: str, fitted_clf, expanded_names):
    """
    Returns pd.Series indexed by expanded_names
    """
    if model_name == "LR":
        coef = fitted_clf.coef_.ravel()
        imp = np.abs(coef)
        return pd.Series(imp, index=expanded_names)

    if model_name == "RF":
        return pd.Series(fitted_clf.feature_importances_, index=expanded_names)

    if model_name == "XGBoost":
        return pd.Series(fitted_clf.feature_importances_, index=expanded_names)

    if model_name == "CatBoost":
        return pd.Series(fitted_clf.get_feature_importance(), index=expanded_names)

    raise ValueError(f"Unknown model: {model_name}")


# =========================
# 8) Aggregate expanded -> original + direction
# =========================
def aggregate_to_original(imp_expanded: pd.Series,
                          shap_values_2d: np.ndarray,
                          expanded_names: list,
                          group_map: dict):
    """
    Returns df_agg (original-variable level):
      FeatureRaw (original feature name),
      Importance (sum over expanded),
      MeanAbsSHAP (sum over expanded mean(|shap|)),
      MeanSHAP (sum over expanded mean(shap))
    """
    mean_abs_shap_exp = np.mean(np.abs(shap_values_2d), axis=0)  # (p,)
    mean_shap_exp = np.mean(shap_values_2d, axis=0)              # (p,)

    df_long = pd.DataFrame({
        "ExpandedFeature": expanded_names,
        "FeatureRaw": [group_map[n] for n in expanded_names],
        "Importance": imp_expanded.values,
        "MeanAbsSHAP": mean_abs_shap_exp,
        "MeanSHAP": mean_shap_exp,
    })

    df_agg = (df_long
              .groupby("FeatureRaw", as_index=False)
              .agg({"Importance": "sum", "MeanAbsSHAP": "sum", "MeanSHAP": "sum"}))

    return df_agg


def direction_for_feature(feature_raw: str,
                          X_raw: pd.DataFrame,
                          shap_values_2d: np.ndarray,
                          expanded_names: list,
                          group_map: dict,
                          num_vars: list):
    """
    Direction rule:
      - numeric: Spearman corr between raw feature values and aggregated SHAP contribution -> + / - / mixed
      - categorical: sign of mean aggregated SHAP contribution -> + / - / mixed
    """
    idxs = [i for i, nm in enumerate(expanded_names) if group_map[nm] == feature_raw]
    if len(idxs) == 0:
        return "mixed"

    shap_feat = shap_values_2d[:, idxs].sum(axis=1)  # aggregated shap contribution

    # numeric
    if feature_raw in num_vars:
        x = pd.to_numeric(X_raw[feature_raw], errors="coerce").values
        mask = np.isfinite(x) & np.isfinite(shap_feat)
        if mask.sum() < 10:
            return "mixed"

        x_m = x[mask]
        s_m = shap_feat[mask]

        # if constant -> no correlation defined
        if np.nanstd(x_m) == 0 or np.nanstd(s_m) == 0:
            return "mixed"

        rho, _ = spearmanr(x_m, s_m)
        if np.isnan(rho):
            return "mixed"
        return "+" if rho > 0 else "-" if rho < 0 else "mixed"

    # categorical (one-hot aggregated): mean sign
    m = np.nanmean(shap_feat)
    if m > 0:
        return "+"
    if m < 0:
        return "-"
    return "mixed"


# =========================
# 9) Ordering function (your requested order)
# =========================
def apply_display_and_order(df_out: pd.DataFrame):
    """
    df_out has FeatureRaw -> map to Feature (display) and order by FEATURE_ORDER
    """
    df_out["Feature"] = df_out["FeatureRaw"].map(DISPLAY_NAME).fillna(df_out["FeatureRaw"])
    df_out["Feature"] = df_out["Feature"].astype(FEATURE_ORDER_TYPE)
    df_out = df_out.sort_values("Feature").reset_index(drop=True)
    return df_out


# =========================
# 10) Main runner
# =========================
def run_supp_tables(df: pd.DataFrame,
                    seed=42,
                    test_size=0.2,
                    topk=None,
                    save_prefix="suppTable_featureImp_SHAP_dir"):
    _check_columns(df)

    y_all = df[TARGET].astype(int).values
    idx = np.arange(len(df))

    idx_train, idx_test = train_test_split(
        idx, test_size=test_size, stratify=y_all, random_state=seed
    )

    models = get_models(seed)

    all_orig = []
    all_expanded = []

    for step_name, feats in MODEL_FEATURES.items():
        X = df[feats].copy()
        X_train = X.iloc[idx_train].copy()
        X_test = X.iloc[idx_test].copy()
        y_train = y_all[idx_train]
        y_test = y_all[idx_test]  # not used here, but kept for reference

        preprocess = build_preprocess(X_train)
        preprocess.fit(X_train)

        expanded_names, group_map, num_vars, cat_vars = get_feature_names_and_groups(preprocess, X_train)

        Xtr = preprocess.transform(X_train)
        Xte = preprocess.transform(X_test)

        Xtr_dense = _to_dense(Xtr)
        Xte_dense = _to_dense(Xte)

        for model_name, estimator in models.items():
            clf = estimator
            clf.fit(Xtr_dense, y_train)

            # importance at expanded-feature level
            imp_exp = get_importance_expanded(model_name, clf, expanded_names)

            # SHAP at test set (held-out interpretation)
            shap_vals_2d = compute_shap_2d(model_name, clf, Xte_dense, background=Xtr_dense[:min(200, Xtr_dense.shape[0])])

            # orig-level aggregation
            df_agg = aggregate_to_original(
                imp_expanded=imp_exp,
                shap_values_2d=shap_vals_2d,
                expanded_names=expanded_names,
                group_map=group_map
            )

            # direction at orig level
            df_agg["Direction"] = [
                direction_for_feature(
                    feature_raw=fraw,
                    X_raw=X_test,
                    shap_values_2d=shap_vals_2d,
                    expanded_names=expanded_names,
                    group_map=group_map,
                    num_vars=num_vars
                )
                for fraw in df_agg["FeatureRaw"].values
            ]

            # add identifiers
            df_agg.insert(0, "Model", model_name)
            df_agg.insert(0, "Step", step_name)

            # map display names + apply your order
            df_agg = apply_display_and_order(df_agg)

            # keep only columns needed for supplementary table
            df_agg_out = df_agg[["Step", "Model", "Feature", "Importance", "MeanAbsSHAP", "MeanSHAP", "Direction"]].copy()

            # optional top-k AFTER ordering (if you want top-k, you probably want by SHAP instead;
            # but user requested fixed order, so we keep fixed order and then cut.)
            if topk is not None:
                df_agg_out = df_agg_out.head(topk).copy()

            # save per (step, model) origlevel
            out_path = os.path.join(OUT_DIR, f"{save_prefix}_{step_name}_{model_name}_origlevel.csv")
            df_agg_out.to_csv(out_path, index=False)

            all_orig.append(df_agg_out)

            # expanded-level detail (no Direction by design)
            df_exp = pd.DataFrame({
                "Step": step_name,
                "Model": model_name,
                "ExpandedFeature": expanded_names,
                "FeatureRaw": [group_map[n] for n in expanded_names],
                "Feature": [DISPLAY_NAME.get(group_map[n], group_map[n]) for n in expanded_names],
                "Importance": imp_exp.values,
                "MeanAbsSHAP": np.mean(np.abs(shap_vals_2d), axis=0),
                "MeanSHAP": np.mean(shap_vals_2d, axis=0),
            })

            # optional: sort expanded detail by MeanAbsSHAP descending for debugging
            df_exp = df_exp.sort_values("MeanAbsSHAP", ascending=False).reset_index(drop=True)

            out_path2 = os.path.join(OUT_DIR, f"{save_prefix}_{step_name}_{model_name}_expandedlevel.csv")
            df_exp.to_csv(out_path2, index=False)

            all_expanded.append(df_exp)

    df_all_orig = pd.concat(all_orig, ignore_index=True)
    df_all_exp = pd.concat(all_expanded, ignore_index=True)

    # sort combined orig by Step, Model, and your feature order
    df_all_orig["Feature"] = df_all_orig["Feature"].astype(FEATURE_ORDER_TYPE)
    df_all_orig = df_all_orig.sort_values(["Step", "Model", "Feature"]).reset_index(drop=True)

    # save combined
    df_all_orig.to_csv(os.path.join(OUT_DIR, f"{save_prefix}_ALL_models_steps_origlevel.csv"), index=False)
    df_all_exp.to_csv(os.path.join(OUT_DIR, f"{save_prefix}_ALL_models_steps_expandedlevel.csv"), index=False)

    return df_all_orig, df_all_exp


# =========================
# 11) RUN
# =========================
# df must already exist in your environment
df_supp, df_supp_expanded = run_supp_tables(
    df,
    seed=SEED,
    test_size=TEST_SIZE,
    topk=TOPK,
    save_prefix=SAVE_PREFIX
)

print("Saved files under:", OUT_DIR)
print("Combined orig-level file:", os.path.join(OUT_DIR, f"{SAVE_PREFIX}_ALL_models_steps_origlevel.csv"))


## Figure 4: SHAP summary plot for random forest Step 3


In [ ]:
def plot_shap_summary_rf_step3(df, save_path=Path("outputs/Figure4_SHAP_RF_Step3.png"), max_display=15):
    feats = MODEL_FEATURES["Step3"]
    X = df[feats].copy()
    y = df[TARGET].astype(int).to_numpy()
    idx = np.arange(len(df))
    idx_train, idx_test = train_test_split(
        idx, test_size=TEST_SIZE, stratify=y, random_state=SEED
    )

    X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
    y_train = y[idx_train]
    preprocess = build_preprocess(X_train)
    pipeline = Pipeline([
        ("prep", preprocess),
        ("clf", get_models(SEED)["RF"]),
    ])
    pipeline.fit(X_train, y_train)

    prep = pipeline.named_steps["prep"]
    fitted_rf = pipeline.named_steps["clf"]
    X_test_transformed = _to_dense(prep.transform(X_test))
    expanded_names, _, _, _ = get_feature_names_and_groups(prep, X_train)

    explainer = shap.TreeExplainer(fitted_rf)
    shap_values = _ensure_shap_2d(explainer.shap_values(X_test_transformed))

    shap.summary_plot(
        shap_values,
        X_test_transformed,
        feature_names=expanded_names,
        max_display=max_display,
        show=False,
    )
    plt.title("SHAP summary plot (Random forest, Step 3)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    return pipeline


rf_step3_pipeline = plot_shap_summary_rf_step3(df)
print("Saved Figure 4 to outputs/Figure4_SHAP_RF_Step3.png")


## Reproducibility note

This public notebook intentionally excludes patient-level data and stored patient-level predictions. The code specifies the complete predictor sets, preprocessing, fixed model settings, random split, five-fold cross-validation, calibration, decision-curve, explainability, and temporal-validation procedures used in the study.
